In [ ]:
import pandas as pd
import json
import numpy as np


In [ ]:
def compare_algorithms(sequence_name, file_locs, algorithm_names):
    """
    Compares the average execution time and average time to first result
    for a specific sequence across multiple algorithms.
    """
    results = []

    for file_loc, algo_name in zip(file_locs, algorithm_names):
        try:
            # Read the file
            df = pd.read_csv(file_loc, sep=';')

            # Filter for the chosen sequence and successful executions
            seq_df = df[(df['name'] == sequence_name) & (df['error'] == False)]

            if seq_df.empty:
                results.append({
                    'Algorithm': algo_name,
                    'Avg Execution Time (ms)': None,
                    'Avg Time to First Result (ms)': None
                })
                continue

            # Calculate overall average execution time
            avg_exec_time = seq_df['time'].mean()

            # Calculate overall average time to first result
            first_results = []
            for timestamps_str in seq_df['timestampsAll']:
                if pd.isna(timestamps_str):
                    continue
                try:
                    # Parse the string representation of lists
                    timestamps_list = json.loads(timestamps_str)

                    # Extract the first result of each run
                    for run in timestamps_list:
                        if len(run) > 0:
                            first_results.append(run[0])
                except json.JSONDecodeError:
                    pass

            avg_first_res_time = np.mean(first_results) if first_results else None

            # Store the data
            results.append({
                'Algorithm': algo_name,
                'Avg Execution Time (ms)': avg_exec_time,
                'Avg Time to First Result (ms)': avg_first_res_time
            })

        except Exception as e:
             results.append({
                 'Algorithm': algo_name,
                 'Avg Execution Time (ms)': str(e),
                 'Avg Time to First Result (ms)': str(e)
             })

    return pd.DataFrame(results)

In [ ]:
file_locations = ['query-times.csv']
algorithm_names = ['Baseline Algorithm']
sequence_to_compare = 'sequence_7'

comparison_df = compare_algorithms(sequence_to_compare, file_locations, algorithm_names)
print(comparison_df)